# 05 · Central midfielder and winger scouting profiles

Extend the recruitment project to two additional roles, using the same event cohort and elapsed participation-time convention as the striker notebook. These descriptive profiles support scouting questions and video review, not validated transfer recommendations.

Run from your project's notebooks folder. No extra packages are required. By default we reuse the event files already downloaded in notebook 04. The full World Cup should give 64 matches; this is still a short international sample rather than a club-season recruitment cohort.


In [ ]:
import sys
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel."
PROJECT_OVERRIDE = None  # Optional: Path("/path/to/soccer-recruitment")
if PROJECT_OVERRIDE is not None:
    candidates = [Path(PROJECT_OVERRIDE).expanduser()]
else:
    candidates = []
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([folder, folder / "soccer-recruitment"])
ROOT = next((p.resolve() for p in candidates if (p / "src/soccer/cli.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory, or set PROJECT_OVERRIDE to your complete checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for folder in ("reports", "data/processed"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", ROOT.name)

paths = sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
if not paths:
    raise FileNotFoundError("Run notebook 01 with acquisition enabled first.")
print("Downloaded event matches:", len(paths))


In [ ]:
import sys
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from soccer.acquire import fetch

EXPAND_COHORT = False
MIN_MINUTES = 300  # Illustrative eligibility threshold; not a guarantee of reliability.
MIN_STRIKER_EVENT_SHARE = 0.5
if EXPAND_COHORT:
    fetch(ROOT, competition=43, season=106, match_count=64, game=1)
else:
    print("Using existing event files only.")


In [ ]:
catalog = json.loads((ROOT / "data/raw/statsbomb/matches/43_106.json").read_text(encoding="utf-8"))
match_ids = {str(m["match_id"]) for m in catalog}
events_by_match = {p.stem: json.loads(p.read_text(encoding="utf-8"))
                   for p in sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
                   if p.stem in match_ids}
print(f"Analyzing {len(events_by_match)} of {len(match_ids)} catalog matches.")
print("Events:", sum(map(len, events_by_match.values())))
if not events_by_match:
    raise ValueError("No eligible match files. Enable cohort expansion.")
if EXPAND_COHORT and len(events_by_match) != len(match_ids):
    raise ValueError("Incomplete tournament download; inspect missing files.")


## 1. Audit participation time

Elapsed participation includes stoppage time and temporary injury absences, excludes halftime breaks and shootouts, and accounts for substitutions and on-field dismissals. Bench discipline is logged without inventing minutes. The calculation below is self-contained and matches the corrected striker notebook.


In [ ]:
def seconds(timestamp):
    h, m, s = timestamp.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def playing_time(events, roster_ids):
    played_periods = sorted({e["period"] for e in events if 1 <= e["period"] <= 4})
    if played_periods != list(range(1, max(played_periods)+1)):
        raise ValueError("Missing played period")
    durations = {}
    offsets = {}
    for period in played_periods:
        ends = [seconds(e["timestamp"]) for e in events if e["period"] == period and e["type"]["name"] == "Half End"]
        if not ends:
            raise ValueError(f"Period {period} has no Half End")
        offsets[period] = sum(durations.values())
        durations[period] = max(ends)
    total = sum(durations.values())
    active, rows, intervals = {}, {}, []
    removed = set()
    temporary_off = Counter()
    red_cards = Counter()
    permanent_off = Counter()
    bench_discipline_events = 0

    def ensure(team, player):
        key = (team["id"], player["id"])
        if key not in roster_ids:
            raise ValueError(f"Player absent from lineup roster: {key}")
        rows.setdefault(key, dict(team_id=team["id"], team=team["name"], player_id=player["id"], player=player["name"], minutes=0.0))
        return key

    def close(key, t, reason):
        if key not in active:
            raise ValueError(f"{reason}: player not active: {key}")
        start = active.pop(key)
        if t < start:
            raise ValueError("Negative participation interval")
        rows[key]["minutes"] += (t-start)/60
        intervals.append((key, start, t, reason))
        removed.add(key)

    for e in events:
        if e["type"]["name"] == "Starting XI":
            for slot in e["tactics"]["lineup"]:
                key = ensure(e["team"], slot["player"])
                if key in active:
                    raise ValueError("Duplicate starter")
                active[key] = 0.0
    teams = {key[0] for key in active}
    if len(teams) != 2 or any(sum(k[0] == team for k in active) != 11 for team in teams):
        raise ValueError("Expected exactly two Starting XIs of eleven players")
    ordered = sorted((e for e in events if e["period"] in durations), key=lambda e: (e["period"], seconds(e["timestamp"]), e.get("index", 0)))
    for e in ordered:
        t = offsets[e["period"]] + seconds(e["timestamp"])
        if seconds(e["timestamp"]) > durations[e["period"]] + 0.001:
            raise ValueError("Event occurs after period end")
        kind = e["type"]["name"]
        if kind == "Substitution":
            outgoing = ensure(e["team"], e["player"])
            # A prior permanent-departure event may already have ended this player's interval.
            if outgoing in active:
                close(outgoing, t, "substitution")
            elif outgoing not in removed:
                raise ValueError("Outgoing substitute never active")
            incoming = ensure(e["team"], e["substitution"]["replacement"])
            if incoming in active or incoming in removed:
                raise ValueError("Unsupported substitute re-entry")
            active[incoming] = t
        if kind == "Player Off":
            key = ensure(e["team"], e["player"])
            if e.get("player_off", {}).get("permanent", False):
                permanent_off[key[0]] += 1
                if key in active:
                    close(key, t, "permanent departure")
            else:
                temporary_off[key] += 1
        if kind == "Bad Behaviour" and e.get("position", {}).get("name") == "Substitute":
            bench_discipline_events += 1
            continue  # Bench discipline does not create participation time or an on-field dismissal.
        card = e.get("foul_committed", {}).get("card", {}).get("name") or e.get("bad_behaviour", {}).get("card", {}).get("name")
        if card in ("Red Card", "Second Yellow"):
            key = ensure(e["team"], e["player"])
            if key in active:
                close(key, t, card)
                red_cards[key[0]] += 1
            else:
                raise ValueError(f"Dismissal of inactive player needs manual review: {key}")
    for key in list(active):
        close(key, total, "match end")
    for key, row in rows.items():
        row["temporary_off_events"] = temporary_off[key]
        if not 0 <= row["minutes"] <= total/60 + 1e-8:
            raise ValueError("Invalid minutes")
    audit = dict(match_elapsed_minutes=total/60, period_minutes={str(p):d/60 for p,d in durations.items()},
                 temporary_off_events=sum(temporary_off.values()), dismissals=sum(red_cards.values()),
                 permanent_departures=sum(permanent_off.values()), participants=len(rows), bench_discipline_events=bench_discipline_events)
    for team in teams:
        actual = sum(r["minutes"] for k,r in rows.items() if k[0] == team)
        expected = 11*total/60
        if actual > expected + 1e-6:
            raise ValueError("Team minutes exceed eleven-player maximum")
        if not red_cards[team] and not permanent_off[team] and abs(actual-expected) > 1e-6:
            raise ValueError("Team participation time does not reconcile")
    return rows, audit


In [ ]:
minutes_by_match, audits = {}, {}
for match_id, events in events_by_match.items():
    roster = json.loads((ROOT / f"data/raw/statsbomb/lineups/{match_id}.json").read_text(encoding="utf-8"))
    roster_ids = {(team["team_id"], p["player_id"]) for team in roster for p in team["lineup"]}
    minutes_by_match[match_id], audits[match_id] = playing_time(events, roster_ids)
print("Audited matches:", len(audits))
print("Temporary departure events (included in participation time):", sum(a["temporary_off_events"] for a in audits.values()))


## 2. Make role and feature definitions explicit

Default CM roles: Center Midfield, Left Center Midfield, Right Center Midfield. Defensive midfield positions can be included deliberately with `INCLUDE_DEFENSIVE_MIDFIELD=True`. Attacking midfield and wing backs remain excluded. Wingers are Left Wing and Right Wing; wide midfielders can optionally be included. Formation differences make these choices material.

Eligibility requires at least 300 elapsed minutes and at least half of positioned events in the selected role. These are illustrative filters. Role share is event-weighted, not time-weighted; features and minutes include all roles played. Borderline cases need manual review.

| Measure | Definition and interpretation |
|---|---|
| Completed forward passes (10 units) | Completed live-ball passes with end x minus start x ≥ 10 on the StatsBomb 120×80 grid. A custom forward-progression proxy, not a provider-standard progressive-pass metric. |
| Final-third entries | Completed live-ball passes starting at x < 80 and ending at x ≥ 80. Counts entries, not all passes inside that third. |
| Forward carries (10 units) | Carries with end x minus start x ≥ 10 grid units. Does not establish beating an opponent. |
| Box entries | Completed live-ball passes or carries starting outside and ending inside x ≥ 102, 18 ≤ y ≤ 62. End position does not prove control or shot creation. |
| Non-penalty xG assisted | Sum of non-penalty shot xG linked to the player's passes by shot `key_pass_id`. This is a shot-based chance-creation proxy, not a separately fitted xA model. |
| Dribble success | Successful dribble events divided by all dribble attempts. Small denominators remain visible. |
| Pressures / interceptions | Event counts, without judging success or tactical appropriateness. |
| Miscontrols / dispossessions | Separate recorded event counts, not a comprehensive possession-loss metric. |

Live-ball passes here exclude named Corner, Free Kick, Throw-in, Kick Off and Goal Kick pass types. Shot assists include all pass situations; npxG assisted includes non-penalty shots from all situations. Coordinate units are **not meters**. Metrica tracking is not joined to these named players.


In [ ]:
import math
INCLUDE_DEFENSIVE_MIDFIELD = False
INCLUDE_WIDE_MIDFIELD = False
MIN_MINUTES = 300
MIN_ROLE_EVENT_SHARE = 0.5
ROLE_GROUPS = {
    "CM": {"Center Midfield", "Left Center Midfield", "Right Center Midfield"},
    "Winger": {"Left Wing", "Right Wing"},
}
if INCLUDE_DEFENSIVE_MIDFIELD:
    ROLE_GROUPS["CM"].update({"Center Defensive Midfield", "Left Defensive Midfield", "Right Defensive Midfield"})
if INCLUDE_WIDE_MIDFIELD:
    ROLE_GROUPS["Winger"].update({"Left Midfield", "Right Midfield"})
DEAD_BALL_TYPES = {"Corner", "Free Kick", "Throw-in", "Kick Off", "Goal Kick"}

def valid_location(loc):
    return isinstance(loc, (list, tuple)) and len(loc) >= 2 and all(isinstance(v, (int,float)) and math.isfinite(v) for v in loc[:2])

def inside_box(loc):
    return valid_location(loc) and 102 <= loc[0] <= 120 and 18 <= loc[1] <= 62

def forward_10(start, end):
    return valid_location(start) and valid_location(end) and end[0] - start[0] >= 10

METRICS = ["passes", "completed_passes", "completed_forward_passes_10u", "final_third_pass_entries",
           "forward_carries_10u", "box_pass_entries", "box_carry_entries", "shot_assists", "npxg_assisted",
           "dribble_attempts", "successful_dribbles", "nonpen_shots", "npxg", "pressures", "interceptions",
           "miscontrols", "dispossessions"]
players, excluded_bench, unmatched_shot_links = {}, [], []
for match_id, events in events_by_match.items():
    passes_by_id = {e["id"]:e for e in events if e["type"]["name"] == "Pass" and e["period"] != 5}
    for key, exposure in minutes_by_match[match_id].items():
        row = players.setdefault(key, dict(team_id=key[0], team=exposure["team"], player_id=key[1], player=exposure["player"],
            matches=0, minutes=0.0, positioned_events=0, CM_events=0, Winger_events=0, **{m:0 for m in METRICS}))
        row["matches"] += 1
        row["minutes"] += exposure["minutes"]
    for e in events:
        if e["period"] == 5 or "player" not in e:
            continue
        kind = e["type"]["name"]
        key = (e["team"]["id"],e["player"]["id"])
        if kind == "Bad Behaviour" and e.get("position",{}).get("name") == "Substitute":
            excluded_bench.append(dict(match_id=match_id, event_id=e.get("id"), player=e["player"]["name"]))
            continue
        if key not in minutes_by_match[match_id]:
            raise ValueError(f"Unmatched playing action: {match_id}, {key}, {kind}")
        row = players[key]
        role = e.get("position",{}).get("name")
        if role:
            row["positioned_events"] += 1
            for group, roles in ROLE_GROUPS.items():
                row[group+"_events"] += int(role in roles)
        start = e.get("location")
        if kind == "Pass":
            info = e["pass"]
            complete = "outcome" not in info
            end = info.get("end_location")
            live = info.get("type",{}).get("name") not in DEAD_BALL_TYPES
            row["passes"] += 1
            row["completed_passes"] += int(complete)
            row["shot_assists"] += int(info.get("shot_assist",False))
            if complete and live and valid_location(start) and valid_location(end):
                row["completed_forward_passes_10u"] += int(forward_10(start,end))
                row["final_third_pass_entries"] += int(start[0] < 80 <= end[0])
                row["box_pass_entries"] += int(not inside_box(start) and inside_box(end))
        elif kind == "Carry":
            end = e["carry"].get("end_location")
            row["forward_carries_10u"] += int(forward_10(start,end))
            if valid_location(start) and valid_location(end):
                row["box_carry_entries"] += int(not inside_box(start) and inside_box(end))
        elif kind == "Dribble":
            row["dribble_attempts"] += 1
            row["successful_dribbles"] += int(e["dribble"]["outcome"]["name"] == "Complete")
        elif kind == "Shot" and e["shot"]["type"]["name"] != "Penalty":
            xg = e["shot"].get("statsbomb_xg",0.0)
            row["nonpen_shots"] += 1
            row["npxg"] += xg
            link = e["shot"].get("key_pass_id")
            if link:
                assist = passes_by_id.get(link)
                if assist is None:
                    unmatched_shot_links.append(dict(match_id=match_id, shot_id=e["id"], key_pass_id=link))
                else:
                    assist_key = (assist["team"]["id"], assist["player"]["id"])
                    if assist_key not in minutes_by_match[match_id] or assist_key[0] != key[0]:
                        raise ValueError("Invalid shot-assist player/team link")
                    players[assist_key]["npxg_assisted"] += xg
        elif kind in {"Pressure", "Interception", "Miscontrol", "Dispossessed"}:
            metric = {"Pressure":"pressures", "Interception":"interceptions", "Miscontrol":"miscontrols", "Dispossessed":"dispossessions"}[kind]
            row[metric] += 1

profiles = []
for key,row in sorted(players.items()):
    for metric in METRICS:
        row[metric+"_per90"] = 90*row[metric]/row["minutes"] if row["minutes"] > 0 else None
    row["pass_completion_pct"] = 100*row["completed_passes"]/row["passes"] if row["passes"] else None
    row["dribble_success_pct"] = 100*row["successful_dribbles"]/row["dribble_attempts"] if row["dribble_attempts"] else None
    for group in ROLE_GROUPS:
        row[group+"_event_share"] = row[group+"_events"]/row["positioned_events"] if row["positioned_events"] else 0.0
        row[group+"_eligible"] = row["minutes"] >= MIN_MINUTES and row[group+"_event_share"] >= MIN_ROLE_EVENT_SHARE
    profiles.append(row)
print("Profiles:",len(profiles), "| bench discipline excluded:",len(excluded_bench), "| unmatched shot links:",len(unmatched_shot_links))


## 3. Compare within roles, with denominators visible

For CMs, inspect forward passing, final-third entries, npxG assisted and defensive involvement. For wingers, inspect dribbles, box entries, npxG and chance creation. Display alphabetically; do not combine these measures into a single score without a recruitment brief and defensible weights.

Empty eligibility results in the three-match starter are expected. Use notebook 04's full-tournament acquisition rather than quietly lowering thresholds. Even 300 minutes is a small sample. High completion may reflect safer choices; high pressures may reflect less possession; tactical assignments affect every metric.


In [ ]:
for group in ROLE_GROUPS:
    eligible = sorted([r for r in profiles if r[group+"_eligible"]],key=lambda r:(r["team"],r["player"]))
    print(f"\n{group}: {len(eligible)} eligible; minimum {MIN_MINUTES} minutes and {MIN_ROLE_EVENT_SHARE:.0%} role-event share")
    selected = (["completed_forward_passes_10u_per90", "final_third_pass_entries_per90", "npxg_assisted_per90"]
                if group == "CM" else ["successful_dribbles_per90", "box_carry_entries_per90", "npxg_per90"])
    for r in eligible[:20]:
        values = " | ".join(f"{m}: {r[m]:.2f}" for m in selected)
        print(f"{r['player']} ({r['team']}, {r['minutes']:.1f} min) | {values}")


## 4. Validate definitions, reconcile counts and export

The small geometry checks verify direction, threshold boundaries and penalty-area membership. Arithmetic reconciles raw counts, rate identities and shot-linked npxG assisted. Missing shot links are saved as a quality issue; they reduce measured chance creation and require review.


In [ ]:
assert forward_10([50,40],[60,40])
assert not forward_10([60,40],[50,40])
assert not forward_10([50,40],[59.9,40])
assert not forward_10(None,[60,40])
assert inside_box([102,18]) and not inside_box([101.9,40])
assert not inside_box([110,63])
for kind,metric in [("Pass","passes"),("Dribble","dribble_attempts"),("Pressure","pressures"),("Interception","interceptions")]:
    expected = sum(e["type"]["name"] == kind and "player" in e and e["period"] != 5 for events in events_by_match.values() for e in events)
    assert sum(r[metric] for r in profiles) == expected
expected_assisted_xg = 0.0
for events in events_by_match.values():
    pass_ids = {e["id"] for e in events if e["type"]["name"] == "Pass" and e["period"] != 5}
    expected_assisted_xg += sum(e["shot"].get("statsbomb_xg",0) for e in events if e["type"]["name"] == "Shot" and e["period"] != 5 and e["shot"]["type"]["name"] != "Penalty" and e["shot"].get("key_pass_id") in pass_ids)
assert math.isclose(sum(r["npxg_assisted"] for r in profiles),expected_assisted_xg)
for r in profiles:
    assert r["successful_dribbles"] <= r["dribble_attempts"]
    assert r["completed_forward_passes_10u"] <= r["completed_passes"] <= r["passes"]
    if r["minutes"] > 0:
        assert math.isclose(r["npxg_assisted_per90"]*r["minutes"]/90,r["npxg_assisted"])
for name, output_rows in [("cm_winger_all_profiles.csv",profiles)]+[(g.lower()+"_eligible_profiles.csv",[r for r in profiles if r[g+"_eligible"]]) for g in ROLE_GROUPS]:
    output = ROOT / "reports" / name
    with output.open("w",newline="",encoding="utf-8") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(profiles[0]))
        writer.writeheader()
        writer.writerows(output_rows)
    print("Saved:",output)
manifest=dict(analyzed_match_ids=sorted(events_by_match), role_groups={g:sorted(v) for g,v in ROLE_GROUPS.items()},
              min_minutes=MIN_MINUTES,min_role_event_share=MIN_ROLE_EVENT_SHARE,
              forward_threshold_statsbomb_x_units=10,excluded_bench_events=excluded_bench,unmatched_shot_links=unmatched_shot_links,
              source_revisions=json.loads((ROOT / "sources.lock.json").read_text(encoding="utf-8"))["revisions"])
(ROOT / "reports/cm_winger_analysis_manifest.json").write_text(json.dumps(manifest,indent=2)+"\n",encoding="utf-8")
print("Definition and reconciliation checks passed.")


## Portfolio interpretation

A defensible project now demonstrates multiple role-specific recruitment views, audited exposure, documented event geometry, source provenance and quality exceptions. It does not yet demonstrate that a model generalizes or that a player is undervalued.

For a case study, select a scouting brief, justify cohort choices, compare a few players within that role, inspect match/video context, and report sensitivity to minutes, role definitions and team possession. Avoid presenting pooled CM/winger/striker percentiles as equal-role ability. Starting at the 50% role threshold, role-event share can bias toward more involved roles and potentially assign more than one role at the boundary.

For later predictive evaluation, compute features only before the scouting date, define a future target, use chronological held-out periods and fit preprocessing on training data. This tournament alone is insufficient for a strong future-season recruitment claim. Metrica remains a separate technical demonstration of tracking features.
